# Exploratory Data Analysis: Forest Cover Type
**Minor Project: Predictive Classification using Random Forest Ensembles (Phase I)**

Dataset: Forest Cover Type (Roosevelt National Forest, Colorado), the UCI / Kaggle *Forest Cover Type Prediction* data. 581,012 cells of 30m x 30m, 54 features, 7 cover types.

Modelling target: **Cottonwood/Willow (Cover_Type 4) vs. all others**, which is strongly imbalanced.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
os.makedirs('eda_figures', exist_ok=True)
CSV = 'forest_cover.csv'

# Load data: use the local CSV if present, otherwise download once and save it as a CSV
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
else:
    from sklearn.datasets import fetch_covtype
    df = fetch_covtype(as_frame=True).frame
    df.to_csv(CSV, index=False)          # <- this is the dataset CSV
    print('Saved dataset to', CSV)
df = df.rename(columns={'Cover_Type': 'Cover_Type'})
print(df.shape)
df.head()

## 1. Structure and data quality

In [ ]:
df.info()
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
NUMERIC = ['Elevation','Aspect','Slope','Horizontal_Distance_To_Hydrology','Vertical_Distance_To_Hydrology',
           'Horizontal_Distance_To_Roadways','Hillshade_9am','Hillshade_Noon','Hillshade_3pm',
           'Horizontal_Distance_To_Fire_Points']
df[NUMERIC].describe().T.round(2)

## 2. Target and class imbalance

In [ ]:
COVER = {1:'Spruce/Fir',2:'Lodgepole Pine',3:'Ponderosa Pine',4:'Cottonwood/Willow',5:'Aspen',6:'Douglas-fir',7:'Krummholz'}
df['Target'] = (df['Cover_Type'] == 4).astype(int)
vc = df['Cover_Type'].value_counts().sort_index()
bc = df['Target'].value_counts().sort_index()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x=[COVER[i] for i in vc.index], y=vc.values, ax=ax[0], palette='viridis')
ax[0].set_title('Original 7 cover types'); ax[0].tick_params(axis='x', rotation=40)
ax[1].bar(['Other (0)','Cottonwood/Willow (1)'], bc.values, color=['#4C72B0','#C44E52'])
ax[1].set_title('Binary target')
for i, v in enumerate(bc.values): ax[1].text(i, v, f'{v:,}', ha='center', va='bottom')
plt.tight_layout(); plt.savefig('eda_figures/01_class_imbalance.png', dpi=300); plt.show()
print(f'Minority share: {bc[1]/len(df)*100:.2f}%  |  imbalance ratio 1:{bc[0]/bc[1]:.0f}')
print(f'Accuracy of a model that always predicts 0: {bc[0]/len(df)*100:.2f}%')

**Finding:** The target class is a tiny fraction of the data, so accuracy is misleading. Use stratified splitting, class weights, and judge the model by precision, recall, F1 and ROC-AUC.

## 3. Feature distributions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for a, c in zip(axes.ravel(), NUMERIC):
    sns.histplot(df[c], bins=40, ax=a, color='#55A868'); a.set_title(c.replace('_',' '), fontsize=9)
    a.set_xlabel(''); a.set_ylabel('')
plt.tight_layout(); plt.savefig('eda_figures/02_distributions.png', dpi=300); plt.show()
print('Skewness:'); print(df[NUMERIC].skew().round(2))

In [ ]:
# Outliers via the IQR rule
rows = []
for c in NUMERIC:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    rows.append((c, ((df[c] < q1-1.5*iqr) | (df[c] > q3+1.5*iqr)).mean()*100))
pd.DataFrame(rows, columns=['Feature','% outliers']).round(2).sort_values('% outliers', ascending=False)

**Finding:** Distance features are right-skewed with long tails. These are real terrain extremes, so they are kept. Random Forest is robust to them and needs no scaling.

## 4. Correlation heatmap

In [ ]:
corr = df[NUMERIC + ['Target']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 7}, linewidths=.4)
plt.title('Correlation heatmap'); plt.tight_layout()
plt.savefig('eda_figures/03_correlation_heatmap.png', dpi=300); plt.show()
print(corr['Target'].drop('Target').sort_values(key=abs, ascending=False).round(3))

**Finding:** The hillshade variables correlate with each other and with Aspect (they are derived from slope and aspect). Correlations with the target are modest, which suggests non-linear relationships that suit trees.

## 5. Relationships with cover type

In [ ]:
tmp = df.sample(min(60000, len(df)), random_state=42).assign(Cover=lambda d: d['Cover_Type'].map(COVER))
plt.figure(figsize=(10, 5))
sns.boxplot(data=tmp, x='Cover', y='Elevation', order=[COVER[i] for i in sorted(COVER)], palette='Set2', fliersize=1)
plt.xticks(rotation=30); plt.title('Elevation by cover type'); plt.tight_layout()
plt.savefig('eda_figures/04_elevation_by_cover.png', dpi=300); plt.show()
print(df.groupby('Target')['Elevation'].mean().round(1))

In [ ]:
wild = [c for c in df.columns if c.startswith('Wilderness_Area')]
names = ['Rawah','Neota','Comanche_Peak','Cache_la_Poudre']
df['Wilderness_Area'] = pd.Series(df[wild].values.argmax(axis=1)).map(dict(enumerate(names))).values
r = df.groupby('Wilderness_Area')['Target'].mean().mul(100).reindex(names)
plt.figure(figsize=(8, 4.5)); sns.barplot(x=r.index, y=r.values, palette='mako')
plt.ylabel('% Cottonwood/Willow'); plt.title('Target rate by wilderness area'); plt.tight_layout()
plt.savefig('eda_figures/05_wilderness_rate.png', dpi=300); plt.show()

In [ ]:
soil = [c for c in df.columns if c.startswith('Soil_Type')]
s = df.groupby(df[soil].values.argmax(axis=1))['Target'].mean().mul(100).sort_values(ascending=False).head(10)
print('Soil types with highest Cottonwood/Willow rate (%):'); print(s.round(2))

## 6. EDA summary and preprocessing plan
- **Data quality:** check the missing and duplicate counts printed in Section 1 (this dataset normally has no nulls).
- **Imbalance:** severe, so use a stratified 80:20 split and `class_weight='balanced_subsample'`, and evaluate with F1, precision, recall and ROC-AUC.
- **Encoding:** wilderness area to one-hot; soil columns are already one-hot.
- **Scaling:** skip, because Random Forest is not distance-based.
- **Key predictors to expect:** elevation, distances to hydrology/roads/fire points, soil type, wilderness area.